In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv("reviews.csv")


# sørg for riktige typer
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["helpful"] = pd.to_numeric(df["helpful"], errors="coerce").fillna(0).astype(int)

# dato: du har både mm/dd og dd/mm i eksempelet, så vi prøver begge
d1 = pd.to_datetime(df["review_date"], errors="coerce", dayfirst=False)
d2 = pd.to_datetime(df["review_date"], errors="coerce", dayfirst=True)
df["review_date"] = d1.fillna(d2)

df.head()

In [ ]:
# Beregn antall ord (enkel måte)
df['word_count'] = df['review_text'].str.split().str.len()

# Fjern ekstreme outliers for bedre visualisering
word_count_99 = df['word_count'].quantile(0.99)
helpful_99 = df['helpful'].quantile(0.99)

df_visual = df[(df['word_count'] <= word_count_99) & (df['helpful'] <= helpful_99)]

# Del ordantall inn i grupper
df_visual['word_group'] = pd.cut(df_visual['word_count'], 
                                 bins=[0, 10, 20, 30, 50, 75, 100, 200, 500],
                                 labels=['0-10', '11-20', '21-30', '31-50', '51-75', '76-100', '101-200', '201+'])

# Beregn gjennomsnittlig helpful per gruppe
avg_helpful_by_words = df_visual.groupby('word_group', observed=True)['helpful'].mean().reset_index()

# Visualiser med begrenset y-akse
fig = px.bar(avg_helpful_by_words, x='word_group', y='helpful',
             title='Gjennomsnittlig helpful score per ordantall',
             labels={'word_group': 'Antall ord i anmeldelsen', 
                    'helpful': 'Gjennomsnittlig helpful votes'})
fig.update_yaxes(range=[0, avg_helpful_by_words['helpful'].max() * 1.1])  # Begrens y-aksen
fig.show()

# Sjekk statistikken
print(avg_helpful_by_words)




In [ ]:
# logistic regression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Definer X og y
X = df["review_text"]
y = df["helpful"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Lag pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("model", LogisticRegression())
])

# Tren modellen
pipeline.fit(X_train, y_train)

# Prediksjon
y_pred = pipeline.predict(X_test)

# Evaluer
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


In [ ]:
# naive bayes

In [ ]:
# random forest?

In [ ]:
# LSTM (neural network)